In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from utils import csv_downloader
# from .notebooks.utils import naming



ImportError: attempted relative import with no known parent package

In [ ]:
df_order = csv_downloader(URL = 'https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/rfm/orders.csv',
                    name = 'orders',
                    path = '../data/rfm')

orders saved in ../data/rfm | shape: (4906, 3)


In [31]:
df_order = pd.read_csv('../data/rfm/orders.csv', parse_dates = ['order_date'])

In [32]:
# RFM
df_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 4906 entries, 0 to 4905
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  4906 non-null   str           
 1   order_date   4906 non-null   datetime64[us]
 2   revenue      4906 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 189.5 KB


In [28]:
fig = px.histogram(data_frame=df_order,
             x = 'revenue',
             title = 'Revenue Distribution',
             nbins = 40
             )
fig.update_layout(
    xaxis_title = 'Revenue',
    yaxis_title = 'Count'
)
fig.show()

## RFM

1. max_date() > Recency 
2. max_date - customer laste day 
3. count() - frequens 
4. SUM  revenue

In [34]:
max_date = df_order['order_date'].max()

In [36]:

rfm = df_order.groupby('customer_id').agg(
    {
     'order_date': lambda date: (max_date  - date.max()).days,
     'customer_id': lambda num: len(num),
     'revenue': lambda rev: rev.sum()
     }
)



```sql
-- the same example with SQL
SELECT 
    customer_id,
    COUNT(customer_od) as Frequency
    max_date - max(order_date) as order_date,
    SUM(revenue) as revenue
FROM df_order
GROUP BY customer_id
```

In [39]:
# rfm.rename({})
column_names = ['Recency','Frequency','Monetary']
rfm.columns = column_names

rfm.head()

,Recency,Frequency,Monetary
customer_id,,,
Abbey O'Reilly DVM,204,6,472
Add Senger,139,3,340
Aden Lesch Sr.,193,4,405
Admiral Senger,131,5,448
Agness O'Keefe,89,9,843


In [40]:
rfm.reset_index(inplace = True)

In [41]:
rfm.head()

,customer_id,Recency,Frequency,Monetary
0,Abbey O'Reilly DVM,204,6,472
1,Add Senger,139,3,340
2,Aden Lesch Sr.,193,4,405
3,Admiral Senger,131,5,448
4,Agness O'Keefe,89,9,843


In [60]:
some_range = [i for i  in range(7)]
print(some_range)
example  = pd.qcut(some_range, 3, labels=["bad", "medium", "good"])
recency  = pd.qcut(some_range, 3, labels=["good", "medium", "bad"])
# example
recency

[0, 1, 2, 3, 4, 5, 6]


['good', 'good', 'good', 'medium', 'medium', 'bad', 'bad']
Categories (3, str): ['good' < 'medium' < 'bad']

In [61]:
rfm['R'] = pd.qcut(rfm['Recency'],4, ['4','3','2','1'])
rfm['F'] = pd.qcut(rfm['Frequency'],4, ['1','2','3','4'])
rfm['M'] = pd.qcut(rfm['Frequency'],4, ['1','2','3','4'])

In [62]:
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M
0,Abbey O'Reilly DVM,204,6,472,3,3,3
1,Add Senger,139,3,340,3,1,1
2,Aden Lesch Sr.,193,4,405,3,2,2
3,Admiral Senger,131,5,448,4,2,2
4,Agness O'Keefe,89,9,843,4,4,4


In [ ]:
pd.qcut(rfm['Frequency'],4, ['4','3','2','1'])

0      2
1      4
2      3
3      3
4      1
      ..
990    2
991    1
992    3
993    4
994    3
Name: Frequency, Length: 995, dtype: category
Categories (4, str): ['4' < '3' < '2' < '1']

In [ ]:
pd.qcut(rfm['Recency'],4,['1','2','3','4'])

0      2
1      2
2      2
3      1
4      1
      ..
990    2
991    2
992    3
993    4
994    4
Name: Recency, Length: 995, dtype: category
Categories (4, str): ['1' < '2' < '3' < '4']

In [65]:
px.histogram(data_frame=rfm,
             x = 'M' 
             )

In [66]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [72]:
fig = make_subplots(
    rows = 1,
    cols = 3,
    # subplot_titles =  ( 'Recencey', 'Frequency', 'Monetory' )
)

fig.add_trace(
    go.Histogram( x  = rfm['Recency']),
    row = 1,
    col = 1
)

fig.add_trace(
    go.Histogram(x=rfm['Frequency']),
    row = 1,
    col = 2
)

fig.add_trace(
    go.Histogram(x=rfm['Monetary']),
    row = 1,
    col = 3
)

fig.update_layout(title = "Recence, Frequency, Monetory",
                  showlegend = False)

fig.update_xaxes(title_text = 'Recency', row = 1, col =1)
fig.update_xaxes(title_text = 'Frequency', row = 1, col =2)
fig.update_xaxes(title_text = 'Monetory', row = 1, col =3)
fig.show()

In [78]:
rfm['RFM_Score'] = rfm['R'].astype(int) + rfm['F'].astype(int) + rfm['M'].astype(int)

rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9
1,Add Senger,139,3,340,3,1,1,5
2,Aden Lesch Sr.,193,4,405,3,2,2,7
3,Admiral Senger,131,5,448,4,2,2,8
4,Agness O'Keefe,89,9,843,4,4,4,12


In [79]:
rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,333
1,Add Senger,139,3,340,3,1,1,5,311
2,Aden Lesch Sr.,193,4,405,3,2,2,7,322
3,Admiral Senger,131,5,448,4,2,2,8,422
4,Agness O'Keefe,89,9,843,4,4,4,12,444


In [87]:
rfm_segments = rfm[['customer_id','RFM_Segment']].groupby('RFM_Segment').count()
rfm_segments.reset_index(inplace=True)

In [93]:
rfm_segments.head()
rfm_segments.rename(columns={'customer_id':'number_of_customers'},inplace=True)

In [94]:
rfm_segments.head()

,RFM_Segment,number_of_customers
0,111,127
1,122,77
2,133,24
3,144,21
4,211,60


In [99]:
px.bar(rfm_segments.sort_values('number_of_customers'),
       y =  'RFM_Segment',
       x =  'number_of_customers',
       orientation = 'h',
       title = 'Number of Customers By RFM Segment'
       )

In [ ]:
# Homework Addc

In [ ]:
def naming(df):
    if df['RFM_Score'] >= 9:
        return 'Can\'t Loose Them'
    elif ((df['RFM_Score'] >= 8) and (df['RFM_Score'] < 9)):
        return 'Champions'
    elif ((df['RFM_Score'] >= 7) and (df['RFM_Score'] < 8)):
        return 'Loyal/Commited'
    elif ((df['RFM_Score'] >= 6) and (df['RFM_Score'] < 7)):
        return 'Potential'
    elif ((df['RFM_Score'] >= 5) and (df['RFM_Score'] < 6)):
        return 'Promising'
    elif ((df['RFM_Score'] >= 4) and (df['RFM_Score'] < 5)):
        return 'Requires Attention'
    else:
        return 'Demands Activation'

In [109]:
rfm['Segment_name'] = rfm.apply(naming,axis=1)
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment,Segment_name
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,333,Can't Loose Them
1,Add Senger,139,3,340,3,1,1,5,311,Promising
2,Aden Lesch Sr.,193,4,405,3,2,2,7,322,Loyal/Commited
3,Admiral Senger,131,5,448,4,2,2,8,422,Champions
4,Agness O'Keefe,89,9,843,4,4,4,12,444,Can't Loose Them


In [113]:
grouped_by = rfm.groupby('Segment_name').agg(
    {
        'Recency': 'mean',
        'Frequency': 'mean',
        'Monetary': ['mean', 'count'],
    }
).round(1)

In [114]:
grouped_by.head()

Recency Frequency Monetary      
                      mean      mean     mean count
Segment_name                                       
Can't Loose Them     177.3       7.5    707.4   300
Champions            161.8       5.0    460.0   128
Demands Activation   591.0       2.2    211.5   127
Loyal/Commited       242.3       4.8    436.5   118
Potential            261.1       4.0    388.1   141

In [115]:
column_names = ['Recency_Mean', 'Frequency_Mean','Monetary_Mean', 'Count']
grouped_by.columns = column_names

grouped_by.reset_index(inplace=True)

grouped_by.head()

,Segment_name,Recency_Mean,Frequency_Mean,Monetary_Mean,Count
0,Can't Loose Them,177.3,7.5,707.4,300
1,Champions,161.8,5.0,460.0,128
2,Demands Activation,591.0,2.2,211.5,127
3,Loyal/Commited,242.3,4.8,436.5,118
4,Potential,261.1,4.0,388.1,141


In [116]:
px.treemap(grouped_by,
           path= ['Segment_name'],
           values = 'Count',
           color = 'Monetary_Mean',
           hover_data= ['Recency_Mean', 'Frequency_Mean','Monetary_Mean'],

           title = 'Segmentation'
           )